# 實驗：DDPM 無條件人臉生成

- 使用 UTKFace dataset（23,708 張人臉圖片）訓練 DDPM 無條件生成模型。
- 影像統一 Resize 到 64×64 以符合 UNet 架構。
- 流程：匯入套件 → Diffusion 類別 → 設定參數 → 建立 DataLoader → 建立模型 → 訓練 → 取樣推理 → TensorBoard 視覺化。

## 1. 匯入套件與工具

In [ ]:
import os
import logging
from pathlib import Path
from types import SimpleNamespace

import torch
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from network import UNet
from utils import get_data, save_images, setup_logging


## 2. 定義 Diffusion 類別

In [ ]:
logging.basicConfig(
    format="%(asctime)s - %(levelname)s: %(message)s",
    level=logging.INFO,
    datefmt="%I:%M:%S",
)


class Diffusion:
    # 管理 DDPM 的加噪排程、訓練取樣時間步，以及反向生成流程
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=256, device="cuda"):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end * (noise_steps / 1000)
        self.img_size = img_size
        self.device = device

        # alpha_hat 是累積保留比例，訓練與採樣都會用它計算目前雜訊強度
        self.beta = self.prepare_noise_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def prepare_noise_schedule(self):
        return torch.linspace(self.beta_start, self.beta_end, self.noise_steps)

    def noise_images(self, x, t):
        # 依照時間步 t 將原圖混入不同強度的 Gaussian noise
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]
        eps = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * eps, eps

    def sample_timesteps(self, n):
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model, n):
        logging.info(f"Sampling {n} new images with DDPM....")
        model.eval()
        with torch.no_grad():
            x = torch.randn((n, 3, self.img_size, self.img_size)).to(self.device)
            for i in tqdm(reversed(range(1, self.noise_steps)), position=0):
                t = (torch.ones(n) * i).long().to(self.device)
                predicted_noise = model(x, t)
                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]
                noise = torch.randn_like(x) if i > 1 else torch.zeros_like(x)

                # 從純雜訊開始逐步扣掉模型預測的 noise，最後還原成圖片
                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise
        x = (x.clamp(-1, 1) + 1) / 2
        x = (x * 255).type(torch.uint8)
        model.train()
        return x

## 3. 設定參數

In [3]:
ROOT = Path.cwd()
DATASET_DIR = ROOT / "UTKFace"

RUN_NAME = "DDPM_UTKFace"
IMAGE_SIZE = 64
BATCH_SIZE = 6
EPOCHS = 50
LR = 3e-4
NOISE_STEPS = 700
NUM_SAMPLE = -1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Dataset:", DATASET_DIR)
print("Device:", DEVICE)

Dataset: /workspace/liangfu/Lab11/UTKFace
Device: cuda


## 4. 建立 DataLoader

In [4]:
# 將 notebook 設定整理成 args，沿用 utils.get_data 的資料載入介面
args = SimpleNamespace(
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    dataset_path=str(DATASET_DIR),
    num_sample=NUM_SAMPLE,
)
dataloader = get_data(args)
print("batches", len(dataloader))

batches 3952


## 5. 建立模型與優化器

In [5]:
setup_logging(RUN_NAME)

# U-Net 學習預測每個時間步加入的 noise，Diffusion 負責加噪與反向採樣
model = UNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
diffusion = Diffusion(img_size=IMAGE_SIZE, device=DEVICE, noise_steps=NOISE_STEPS)
mse = torch.nn.MSELoss()
logger = SummaryWriter(os.path.join("runs", RUN_NAME))

## 6. 訓練迴圈

In [ ]:
best_loss = float("inf")
early_stop_counter = 0
last_epoch_loss = float("inf")
save_every = 10
log_every = 200
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_running_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{EPOCHS}", dynamic_ncols=True)
    for images, _ in pbar:
        images = images.to(DEVICE)

        # 隨機抽時間步並加噪，模型目標是預測當下加入的 noise
        t = diffusion.sample_timesteps(images.shape[0]).to(DEVICE)
        x_t, noise = diffusion.noise_images(images, t)
        predicted_noise = model(x_t, t)
        loss = mse(noise, predicted_noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 使用滑動平均讓進度列的 loss 比單一步驟更穩定
        epoch_running_loss = loss.item() * 0.1 + epoch_running_loss * 0.9
        pbar.set_postfix(MSE=epoch_running_loss)

        if global_step % log_every == 0:
            logger.add_scalar("Loss", epoch_running_loss, global_step)
        global_step += 1

    scheduler.step()

    # 定期保存樣本與權重，方便中途檢查生成品質
    if epoch % save_every == 0:
        sampled_images = diffusion.sample(model, n=16)
        save_images(sampled_images, os.path.join("results", RUN_NAME, f"{epoch}.jpg"))
        torch.save(model.state_dict(), os.path.join("models", RUN_NAME, f"{epoch}.pt"))
        logger.add_images("Samples", sampled_images, epoch)

    if epoch_running_loss < best_loss:
        best_loss = epoch_running_loss
        torch.save(model.state_dict(), os.path.join("models", RUN_NAME, "best.pt"))

    # loss 連續變差時提前停止，避免無效訓練繼續消耗時間
    early_stop_counter = early_stop_counter + 1 if epoch_running_loss > last_epoch_loss else 0
    last_epoch_loss = epoch_running_loss
    if early_stop_counter >= 10:
        print("Loss 連續 10 個 epoch 未下降，提早停止。")
        torch.save(model.state_dict(), os.path.join("models", RUN_NAME, "last.pt"))
        break

logger.close()

## 7. Inference

In [10]:
output_dir = os.path.join("results", RUN_NAME, "samples")
os.makedirs(output_dir, exist_ok=True)

# 載入最佳權重後連續生成多組圖片，方便比較模型輸出穩定度
model.load_state_dict(torch.load(os.path.join("models", RUN_NAME, "best.pt")))

for i in range(10):
    sampled_images = diffusion.sample(model, n=16)
    save_images(sampled_images, os.path.join(output_dir, f"{i:02d}.jpg"))

/tmp/ipykernel_25241/701573579.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join("models", RUN_NAME, "best.pt")))
06:02:15 - 

## 8. TensorBoard

In [9]:
# 開啟 TensorBoard，查看 loss 曲線與採樣圖片
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 227366), started 7:48:32 ago. (Use '!kill 227366' to kill it.)